## Database and SQL for Data Analytics with Python

> In this notebook, we will demosntrate the detailes steps we toom comporising a pipeline that covers the data analytics from raw data source all the way to the SQL output while operating both inside a Python environemnt, and outside.

In [ ]:
#import all the essential libraries and modules
#installing the sql and plot related libraires in the local notebook, since this is a new environemnt.
!pip install ipython-sql prettytable 
import pandas as pd
import sqlite3
import csv
import prettytable

prettytable.DEFAULT = 'DEFAULT'


STEP 1: Connect to the Database 
(SELF_Note: Connection methods of all the three types we learned namely cursoe, pandas, and %sql must be stated here)

In [4]:
connection = sqlite3.connect('sales.db')
%load_ext sql
%sql sqlite:///sales.db

In [5]:
from sqlalchemy import create_engine
engine = create_engine('sqlite:///sales.db')

STEP 2: Read the datasets and convert them into databse tables by creating new tables

In [32]:
df = pd.read_csv(r"C:\Users\DELL\Documents\data-analytics-sql\datasets\superstore.csv")
print (df.shape)
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')
df.head()

(51290, 21)


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,1/1/2011,6/1/2011,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,Office Supplies,Storage,"Tenex Lockers, Blue",408.0,2,0.0,106.140,35.46,Medium,2011
1,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Office Supplies,Supplies,"Acme Trimmer, High Speed",120.0,3,0.1,36.036,9.72,Medium,2011
2,HU-2011-1220,1/1/2011,5/1/2011,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,Office Supplies,Storage,"Tenex Box, Single Width",66.0,4,0.0,29.640,8.17,High,2011
3,IT-2011-3647632,1/1/2011,5/1/2011,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,Office Supplies,Paper,"Enermax Note Cards, Premium",45.0,3,0.5,-26.055,4.82,High,2011
4,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",114.0,5,0.1,37.770,4.70,Medium,2011


### STEP 3: Create a simple DATABASE table out of the original dataset
> We will create one raw table with all the sheet data as it is which we can query later with %sql directly in the notebook.

Then we will create three DIMENSIONAL tables out of the single spreadsheet so to be able to organize the entire data queries more effectively. These three tables will be connected with their Primary keys to the facts table with Foreign keys

Effectively this will internally make a table schema which will be more organised and which we can retrieve and display in our repo 

In [34]:
# option 1: 
original_table = df.to_sql('superstore_original', connection, if_exists = 'replace', index = False) 
original_table

51290

In [35]:
#2 Option:2
original_table = df.to_sql ('superstore_original', engine, if_exists = 'replace', index = False) 
original_table

51290

In [36]:
%%sql 
SELECT * FROM superstore_original LIMIT 2;

 * sqlite:///sales.db
Done.


order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
AG-2011-2040,1/1/2011,6/1/2011,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",408.0,2,0.0,106.14,35.46,Medium,2011
IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",120.0,3,0.1,36.036,9.72,Medium,2011


### Step 4: Create the dimension tables and the facts table (star schema) one by one out of the large dataset in Pandas dataframe

In [37]:
# Three dimensional tables out of the flat spreadsheet. First, we wil create each dimenional table in pandas dataframe and then load them into sql databse tables/

#1. dim_customers
dim_customers = df[['customer_name', 'segment']].drop_duplicates().reset_index(drop = True)
print (dim_customers.shape)
dim_customers.head(5)

(795, 2)


,customer_name,segment
0,Toby Braunhardt,Consumer
1,Joseph Holt,Consumer
2,Annie Thurman,Consumer
3,Eugene Moren,Home Office
4,Magdelene Morse,Consumer


#### Concept of subset in drop_duplicates(): Precautionary Cleaning step

In a messy real-world CSV data, the same product_id can sometimes appear with slightly different product_name spellings due to typos etc. 

- If we do drop_duplicates() on all 4 columns, it would not drop those slightly different misspelled names, considering them unique while they are just the same products_name misspelled (but their ids remain the same as the correctly spalled one). 

- So if we drop the duplicates based on whose id is repeating, not their name or any other repetitive measure, we can ensure all duplicates are really dropped

> subset='product_id' ensures that the deduplication is done only based on the product_id alone.

HENCE here we have to insert a subset='product_id' in the drop_duplicates paranthesis




In [38]:
#2 dim_products 
dim_products = df[['product_id', 'product_name', 'category', 'sub_category']].drop_duplicates(subset='product_id').reset_index(drop = True)   ## subset='product_id' because the same product_id can appear with # slightly different name spellings in raw data — id is the true unique key
print (dim_products.shape)
dim_products.head(5)

(10292, 4)


,product_id,product_name,category,sub_category
0,OFF-TEN-10000025,"Tenex Lockers, Blue",Office Supplies,Storage
1,OFF-SU-10000618,"Acme Trimmer, High Speed",Office Supplies,Supplies
2,OFF-TEN-10001585,"Tenex Box, Single Width",Office Supplies,Storage
3,OFF-PA-10001492,"Enermax Note Cards, Premium",Office Supplies,Paper
4,FUR-FU-10003447,"Eldon Light Bulb, Duo Pack",Furniture,Furnishings


#### dim_locations table: Concept of Creating a primary key

In the source data we can see that location-related attributes have NO natural unique key in the source data

> i) First, we deduplicate the entire table. Since there are no unique keys, we cannot use the precautionary subset in drop_duplicates. It is not super-essential.

> ii) Now that each row is unique, we can generate a surrogate key made of sequential integers 1, 2, 3.

This is to create a unique key which does not exist in original table so as to have a primary key in the dim_locations table which can then be used later to relate to/Join with other tables like 'facts_orders' of the star schema.

Note: len(dim_locations) = number of rows, not string length

In [39]:
#3 dim_locations
dim_locations = df[['state', 'country', 'market', 'region']].drop_duplicates().reset_index(drop=True) #i)
dim_locations.insert(0, 'location_id', range(1, len(dim_locations) + 1)) #ii)

#V.important: Since the dataframe is updated, we need to use the new dataframe fro fact_orders table. For that we need to update the existing dataframe withou touching the original dataframe as that is a reference.
df_updated = df.merge(dim_locations, on=['state', 'country', 'market', 'region'], how='left') #updated dataframe

print (dim_locations.shape)
dim_locations.head()

(1126, 5)


,location_id,state,country,market,region
0,1,Constantine,Algeria,Africa,Africa
1,2,New South Wales,Australia,APAC,Oceania
2,3,Budapest,Hungary,EMEA,EMEA
3,4,Stockholm,Sweden,EU,North
4,5,Ontario,Canada,Canada,Canada


In [40]:
#4 fact_orders table: the table containing all the measurable numbers as well as foreign keys from other three dimesion tables. 

fact_orders = df_updated[['order_id', 'order_date', 'ship_date', 'ship_mode',   #using the uopdated dataframe with the location_id as we need the location_id column as a froeign key
                            'sales', 'quantity', 'discount', 'profit',
                            'shipping_cost', 'order_priority', 'year',
                            'customer_name', 'product_id', 'location_id']]

# dimension tables store descriptive attributes — duplicates mean dirty data
# fact_orders stores transactions — every row IS a unique event
# dropping duplicates here would mean deleting real sales records

print(fact_orders.shape)
fact_orders.head()

(51290, 14)


,order_id,order_date,ship_date,ship_mode,sales,quantity,discount,profit,shipping_cost,order_priority,year,customer_name,product_id,location_id
0,AG-2011-2040,1/1/2011,6/1/2011,Standard Class,408.0,2,0.0,106.140,35.46,Medium,2011,Toby Braunhardt,OFF-TEN-10000025,1
1,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,120.0,3,0.1,36.036,9.72,Medium,2011,Joseph Holt,OFF-SU-10000618,2
2,HU-2011-1220,1/1/2011,5/1/2011,Second Class,66.0,4,0.0,29.640,8.17,High,2011,Annie Thurman,OFF-TEN-10001585,3
3,IT-2011-3647632,1/1/2011,5/1/2011,Second Class,45.0,3,0.5,-26.055,4.82,High,2011,Eugene Moren,OFF-PA-10001492,4
4,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,114.0,5,0.1,37.770,4.70,Medium,2011,Joseph Holt,FUR-FU-10003447,2


### Step 5: Loading the dimensional sub-tables into our sales.db database, to be able to execute SQL queries

In [41]:
dim_locations.to_sql('dim_locations', connection, if_exists='replace', index=False)
dim_customers.to_sql('dim_customers', connection, if_exists='replace', index=False)
dim_products.to_sql('dim_products', connection, if_exists='replace', index=False)
fact_orders.to_sql('fact_orders', connection, if_exists = 'replace', index=False)
print('dimension tables loaded')

dimension tables loaded


## Validation: verify if all the 'star schema' DB tables are successfully loaded: 

In [42]:
%%sql
SELECT
    (SELECT COUNT(*) FROM fact_orders)   AS total_orders,
    (SELECT COUNT(*) FROM dim_customers) AS total_customers,
    (SELECT COUNT(*) FROM dim_products)  AS total_products,
    (SELECT COUNT(*) FROM dim_locations) AS total_locations


 * sqlite:///sales.db
Done.


total_orders,total_customers,total_products,total_locations
51290,795,10292,1126


## Table Schema

In [43]:
%%sql
SELECT * FROM sqlite_master;

 * sqlite:///sales.db
Done.


type,name,tbl_name,rootpage,sql
table,superstore_original,superstore_original,2,"CREATE TABLE superstore_original ( order_id TEXT, order_date TEXT, ship_date TEXT, ship_mode TEXT, customer_name TEXT, segment TEXT, state TEXT, country TEXT, market TEXT, region TEXT, product_id TEXT, category TEXT, sub_category TEXT, product_name TEXT, sales FLOAT, quantity BIGINT, discount FLOAT, profit FLOAT, shipping_cost FLOAT, order_priority TEXT, year BIGINT)"
table,dim_locations,dim_locations,3,"CREATE TABLE ""dim_locations"" (""location_id"" INTEGER, ""state"" TEXT, ""country"" TEXT, ""market"" TEXT, ""region"" TEXT)"
table,dim_customers,dim_customers,2848,"CREATE TABLE ""dim_customers"" (""customer_name"" TEXT, ""segment"" TEXT)"
table,dim_products,dim_products,2850,"CREATE TABLE ""dim_products"" (""product_id"" TEXT, ""product_name"" TEXT, ""category"" TEXT, ""sub_category"" TEXT)"
table,fact_orders,fact_orders,3071,"CREATE TABLE ""fact_orders"" (""order_id"" TEXT, ""order_date"" TEXT, ""ship_date"" TEXT, ""ship_mode"" TEXT, ""sales"" REAL, ""quantity"" INTEGER, ""discount"" REAL, ""profit"" REAL, ""shipping_cost"" REAL, ""order_priority"" TEXT, ""year"" INTEGER, ""customer_name"" TEXT, ""product_id"" TEXT, ""location_id"" INTEGER)"


In [44]:
#We save the relevant table schema into a txt file and store it for user-readability in the 'scripts' folder of the githubrep

schema = pd.read_sql("SELECT name, sql FROM sqlite_master WHERE type='table'", connection)

# Save to scripts folder (one level up, into scripts/ as cleanly separated columns)
with open('../scripts/table_schema_documentation.txt', 'w') as f:
    for _, row in schema.iterrows():
        f.write(f"Table: {row['name']}\n")
        f.write(f"{row['sql']}\n")
        f.write("\n" + "="*50 + "\n\n")
#schema saved!

## Quick SQL Demo
### Demonstrating SQL queries using Python's magic %sql command

In [45]:
%%sql
SELECT * FROM fact_orders LIMIT 5

 * sqlite:///sales.db
Done.


order_id,order_date,ship_date,ship_mode,sales,quantity,discount,profit,shipping_cost,order_priority,year,customer_name,product_id,location_id
AG-2011-2040,1/1/2011,6/1/2011,Standard Class,408.0,2,0.0,106.14,35.46,Medium,2011,Toby Braunhardt,OFF-TEN-10000025,1
IN-2011-47883,1/1/2011,8/1/2011,Standard Class,120.0,3,0.1,36.036,9.72,Medium,2011,Joseph Holt,OFF-SU-10000618,2
HU-2011-1220,1/1/2011,5/1/2011,Second Class,66.0,4,0.0,29.64,8.17,High,2011,Annie Thurman,OFF-TEN-10001585,3
IT-2011-3647632,1/1/2011,5/1/2011,Second Class,45.0,3,0.5,-26.055,4.82,High,2011,Eugene Moren,OFF-PA-10001492,4
IN-2011-47883,1/1/2011,8/1/2011,Standard Class,114.0,5,0.1,37.77,4.7,Medium,2011,Joseph Holt,FUR-FU-10003447,2


In [46]:
%%sql
SELECT 
    segment,
    COUNT(*) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales
FROM fact_orders
JOIN dim_customers ON fact_orders.customer_name = dim_customers.customer_name
GROUP BY segment
ORDER BY total_sales DESC

 * sqlite:///sales.db
Done.


segment,total_orders,total_sales
Consumer,26518,4058118.0
Corporate,15429,2369261.0
Home Office,9343,1407749.0


In [47]:
%%sql
SELECT 
    category,
    ROUND(SUM(profit), 2) AS total_profit
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
GROUP BY category
ORDER BY total_profit DESC

 * sqlite:///sales.db
Done.


category,total_profit
Technology,663778.73
Office Supplies,518473.83
Furniture,286782.25
